In [16]:
from io import BytesIO
from pathlib import Path
import numpy as np
import pandas as pd
import ipywidgets as w
from IPython.display import display
from datasets import load_from_disk
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm
from ultralytics import YOLO

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
ds = load_from_disk(str(ROOT / "data/processed/floor_posture"))
recs = ds.select_columns(["recording"]).to_pandas().recording
first = recs.drop_duplicates()  # index of each recording's first row

# YOLOv8 face detector (single class: FACE)
detector = YOLO(hf_hub_download("arnabdhar/YOLOv8-Face-Detection", "model.pt"))

NAMES = ["Alice", "Bob", "Carol", "Dave", "Erin", "Frank", "Grace", "Heidi", "Ivan", "Judy",
         "Karl", "Laura", "Mallory", "Nina", "Oscar", "Peggy", "Quinn", "Rupert", "Sybil", "Trent",
         "Uma", "Victor", "Wendy", "Xavier", "Yara", "Zed", "Abel", "Bea", "Cyrus", "Dora",
         "Eli", "Fiona", "Gus", "Hana", "Ian"]
assert len(NAMES) == 35 and len(set(NAMES)) == 35

In [ ]:
SAMPLES = 60   # frames scanned per recording
SHOW = 3       # best face crops shown per recording
MIN_CONF = 0.25

def png(im):
    buf = BytesIO()
    im.convert("RGB").resize((150, 150)).save(buf, "PNG")
    return buf.getvalue()

def best_faces(rec_start, n):
    idx = [rec_start + int(k) for k in np.linspace(0, n - 1, SAMPLES)]
    frames = [im.convert("RGB") for im in ds.select(idx).select_columns(["image"])["image"] if im is not None]
    found = []  # (confidence, crop)
    for im, r in zip(frames, detector.predict([np.asarray(f) for f in frames], conf=MIN_CONF, verbose=False)):
        for box in r.boxes:
            x0, y0, x1, y1 = box.xyxy[0].tolist()
            p = 0.2 * max(x1 - x0, y1 - y0)  # a little context around the face
            found.append((float(box.conf), im.crop((x0 - p, y0 - p, x1 + p, y1 + p))))
    return [crop for _, crop in sorted(found, key=lambda t: -t[0])[:SHOW]]

dropdowns = {}
rows = []
for start, rec in tqdm(first.items(), desc="detecting faces"):
    faces = best_faces(start, (recs == rec).sum())
    dd = w.Dropdown(options=["?"] + NAMES, value="?", layout=w.Layout(width="110px"))
    dropdowns[rec] = dd
    rows.append(w.HBox(
        [w.Label(rec, layout=w.Layout(width="330px"))]
        + [w.Image(value=png(f), format="png") for f in faces]
        + ([] if faces else [w.Label("no face detected")])
        + [dd]
    ))
display(w.VBox(rows))

detecting faces: 0it [00:00, ?it/s]

In [19]:
out = ROOT / "data/processed/person_map.csv"  # gitignored
df = pd.DataFrame({
    "recording": list(dropdowns),
    "person": ["" if d.value == "?" else d.value for d in dropdowns.values()],
})
df.to_csv(out, index=False)
print("wrote", out, f"({(df.person != '').sum()}/{len(df)} labelled)")

wrote /Users/sigurdurhaukur/Documents/rug/ai-engineering/data/processed/person_map.csv (27/35 labelled)
